In [2]:
import pandas as pd

In [3]:
swaps = pd.DataFrame(
    [
        ("S1", "ABC", 1000, 1000, 1050, 10),
        ("S2", "ABC",  500, 1000,  960, 10),
        ("S3", "XYZ",  200, 5000, 5000, 25),
        ("S4", "XYZ",  300, 5000, 4900,  0),
    ],
    columns=[
        "swap_id",
        "symbol",
        "quantity",
        "start_price_cents",
        "end_price_cents",
        "dividend_per_share_cents",
    ],
)


In [14]:
swaps

,swap_id,symbol,quantity,start_price_cents,end_price_cents,dividend_per_share_cents,sofr_rate,spread_bps,days,price_return_cents,dividend_amount_cents,equity_amount_cents,notional_cents,financing_rate
0,S1,ABC,1000,1000,1050,10,0.04,150,30,50000,10000,60000,1000000,0.055
1,S2,ABC,500,1000,960,10,0.04,150,30,-20000,5000,-15000,500000,0.055
2,S3,XYZ,200,5000,5000,25,0.04,150,30,0,5000,5000,1000000,0.055
3,S4,XYZ,300,5000,4900,0,0.04,150,30,-30000,0,-30000,1500000,0.055


In [15]:
swaps["sofr_rate"] = 0.04
swaps["spread_bps"] = 150
swaps["days"] = 30

In [10]:
swaps["price_return_cents"] = swaps["quantity"]*(swaps["end_price_cents"]-swaps["start_price_cents"])
swaps["dividend_amount_cents"] = swaps["quantity"]*swaps["dividend_per_share_cents"]
swaps["equity_amount_cents"] = swaps["dividend_amount_cents"] + swaps["price_return_cents"]

In [16]:
swaps["notional_cents"] = swaps["quantity"]*swaps["start_price_cents"]
swaps["financing_rate"] = swaps["sofr_rate"] + .0001*swaps["spread_bps"]

In [17]:
swaps["financing_amount_cents"] = swaps["notional_cents"]*swaps["financing_rate"]*swaps["days"]/360.0

In [18]:
swaps["net_amount_cents"] = swaps["equity_amount_cents"] - swaps["financing_amount_cents"]

In [ ]:
# swaps = swaps.rename(columns={"sofra_rate": "sofr_rate"})

In [26]:
bank = pd.DataFrame(
    [
        ("S1",  55417),
        ("S2", -17242),
        ("S3",    417),
        ("S5",  12500),
    ],
    columns=["swap_id", "net_amount_cents"],
)

In [29]:
two = swaps[["swap_id","net_amount_cents"]]
two

,swap_id,net_amount_cents
0,S1,55416.666667
1,S2,-17291.666667
2,S3,416.666667
3,S4,-36875.000000


In [42]:
reconciled = swaps[["swap_id", "net_amount_cents"]].merge(
    bank,
    on="swap_id",
    how="outer",
    indicator=True,
    suffixes=("_ours", "_bank"), 
    validate="one_to_one")

In [43]:
reconciled

,swap_id,net_amount_cents_ours,net_amount_cents_bank,_merge
0,S1,55416.666667,55417.0,both
1,S2,-17291.666667,-17242.0,both
2,S3,416.666667,417.0,both
3,S4,-36875.000000,NaN,left_only
4,S5,NaN,12500.0,right_only


In [44]:
reconciled["difference_cents"] = reconciled["net_amount_cents_ours"] - reconciled["net_amount_cents_bank"]

In [48]:
reconciled

,swap_id,net_amount_cents_ours,net_amount_cents_bank,_merge,difference_cents
1,S2,-17291.666667,-17242.0,both,-49.666667
3,S4,-36875.000000,NaN,left_only,NaN
4,S5,NaN,12500.0,right_only,NaN


In [47]:
reconciled = reconciled[(reconciled["difference_cents"].abs() > 1.0) | (reconciled["_merge"] != "both")]